# W6 Lab — Multi-Agent Pipeline: Customer Service for a Store

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week06/W6_lab_multiagent.ipynb)

**Goal.** By the end of this lab you can split one job across a pipeline of agents —
planner, reviewer, executor, explainer — each with its own prompt and context, wire
their handoffs as strict JSON, and show with a scored task set that the assembly
works end to end (target ≥ 5/6). The exercises price the reflection stage in calls
and add a new team member.

Why this week: the lecture (notes Ch. 6) argued that a split is bought with handoffs —
explicit, lossy messages — and kept debuggable by per-component evaluation. This lab
runs that argument live on a store's customer-service traffic.

The path: setup → the store's data → tool registry → executor → the pipeline's
agents (plan, reflect ✍️, explain ✍️) → assembly → a measured day of requests →
ablation and a new team member.

Adapted from DeepLearning.AI, *Agentic AI* by Andrew Ng — Module 5 (Multi-Agent
Collaboration), ungraded labs 1–2. The pipeline, agent roles, tool registry, and
planning specification follow ungraded lab 1; the setup is adapted for Google Colab
and the course API standard (`aisuite`, key pasted into the setup cell).

*Runtime:* Google Colab, top-to-bottom, ~80 minutes. Cells marked ✍️ ask for your own writing — a fill-in or a written prediction.

## 1. Setup

### 1.1 Installation

`duckdb` runs SQL over in-memory pandas DataFrames; the executor uses it so tools can
query the store's tables without a database server.

In [ ]:
%pip install -q "aisuite[openai,anthropic]" duckdb

### 1.2 API key and model

Paste your key between the quotes (issuing steps: the API Setup guide on the course
site). The key is yours; do not share the notebook with the key still inside.

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE-YOUR-KEY-HERE"

MODEL = "openai:gpt-4o-mini"   # Anthropic accounts: MODEL = "anthropic:claude-haiku-4-5" and set ANTHROPIC_API_KEY instead

### 1.3 Client and helpers

`chat_text` is the single model-call helper for the whole lab; every agent below is a
function that calls it with its own system prompt. `parse_json_output` recovers a JSON
object from model text: it strips Markdown fences and repairs invalid escape
sequences before parsing (the repair function comes from the source lab).

In [ ]:
import json
import re

import aisuite

client = aisuite.Client()
n_calls = 0

def chat_text(user, system=None):
    """(user text, optional system text) -> assistant reply text; counts calls."""
    global n_calls
    messages = ([{"role": "system", "content": system}] if system else [])
    messages.append({"role": "user", "content": user})
    response = client.chat.completions.create(model=MODEL, messages=messages)
    n_calls += 1
    return response.choices[0].message.content

_ALLOWED_ESC = r'["\\/bfnrtu]'

def _repair_invalid_json_escapes(s):
    """Model JSON with stray backslashes -> parseable string."""
    s = s.replace("\\'", "'")
    return re.sub(rf'\\(?!{_ALLOWED_ESC})', r'', s)

def parse_json_output(text):
    """Model output -> parsed JSON object; strips fences, repairs escapes."""
    stripped = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    try:
        return json.loads(stripped)
    except json.JSONDecodeError:
        return json.loads(_repair_invalid_json_escapes(stripped))

### 1.4 Verification

In [ ]:
print(chat_text("Reply with exactly: ready"))

If the output is `ready`, key and billing work. Any error here is a setup problem, not
a code problem.

## 2. The Store's Data

Two pandas DataFrames stand in for the store's database: an inventory of five
sunglasses models and a transaction ledger opened with the day's register balance.
Both functions come from the source lab.

*Do:* run the cells and skim the three tables — the pipeline's whole world is these rows.


In [ ]:
import random

import pandas as pd

def create_inventory_dataframe():
    """-> DataFrame(name, item_id, description, quantity_in_stock, price) for 5 models."""
    random.seed(42)   # fixed seed: every run starts from the same stock levels
    sunglasses_data = {
        "name": ["Aviator", "Wayfarer", "Mystique", "Sport", "Round"],
        "item_id": ["SG001", "SG002", "SG003", "SG004", "SG005"],
        "description": [
            "Originally designed for pilots, these teardrop-shaped lenses with thin metal frames offer timeless appeal.",
            "Featuring thick, angular frames that make a statement, these sunglasses combine retro charm with modern edge.",
            "Inspired by 1950s glamour, these frames sweep upward at the outer corners to create an elegant silhouette.",
            "Designed for active lifestyles, these wraparound sunglasses feature a single curved lens for maximum coverage.",
            "Circular lenses set in minimalist frames create a thoughtful, artistic appearance.",
        ],
        "quantity_in_stock": [random.randint(3, 25) for _ in range(5)],
        "price": [random.randint(75, 150) for _ in range(5)],
    }
    return pd.DataFrame(sunglasses_data)

def create_transaction_dataframe(opening_balance=500.00):
    """-> DataFrame(transaction_id, customer_name, transaction_summary, transaction_amount, balance_after_transaction)."""
    opening_transaction = {
        "transaction_id": ["TXN001"],
        "customer_name": ["OPENING_BALANCE"],
        "transaction_summary": ["Daily opening register balance"],
        "transaction_amount": [opening_balance],
        "balance_after_transaction": [opening_balance],
    }
    return pd.DataFrame(opening_transaction)

inventory_df = create_inventory_dataframe()
transaction_df = create_transaction_dataframe()

display(inventory_df)
display(transaction_df)

### 2.1 SQL over the DataFrames

`create_duckdb_with_views` registers both DataFrames as SQL views on an in-memory
DuckDB connection, so tools can query them with ordinary SQL.

In [ ]:
import duckdb

def create_duckdb_with_views(inventory_df, transaction_df):
    """(inventory, transactions) -> DuckDB connection with both registered as views."""
    con = duckdb.connect()
    con.register("inventory_df", inventory_df)
    con.register("transaction_df", transaction_df)
    return con

con = create_duckdb_with_views(inventory_df, transaction_df)
display(con.sql("SELECT name, quantity_in_stock, price FROM inventory_df").df())

## 3. The Tool Registry

The pipeline's plans consist only of tool calls — no raw SQL, no free-form code. A
**tool registry** = a dict mapping tool names to functions, so a plan can name a tool
and the executor can look it up (notes Ch. 3: tools; the registry pattern reappears from
the W3 lab). The registry below is the source lab's, in three groups: read tools,
write tools (which return updated DataFrames instead of mutating in place), and
calculation/validation helpers.

In [ ]:
def t_get_inventory_data(con, product_name=None, item_id=None):
    """Lookup by name (case-insensitive) or item_id -> {rows, match_count, item}."""
    if not product_name and not item_id:
        df = con.execute("SELECT * FROM inventory_df").df()   # no filter: browse all
    elif item_id:
        df = con.execute("SELECT * FROM inventory_df WHERE item_id = ?", [item_id]).df()
    else:
        df = con.execute("SELECT * FROM inventory_df WHERE lower(name)=lower(?)",
                         [product_name]).df()
    item = df.iloc[0].to_dict() if len(df) == 1 else None
    return {"rows": df, "match_count": int(len(df)), "item": item}

def t_get_transaction_data(con, mode="last_balance"):
    """mode='last_balance' -> {mode, last_txn_id, last_balance}."""
    if mode == "last_balance":
        df = con.execute(
            "SELECT transaction_id, balance_after_transaction "
            "FROM transaction_df ORDER BY transaction_id DESC LIMIT 1").df()
        last_id = str(df.iloc[0]["transaction_id"]) if not df.empty else None
        last_bal = float(df.iloc[0]["balance_after_transaction"]) if not df.empty else 0.0
        return {"mode": mode, "last_txn_id": last_id, "last_balance": last_bal}
    return {"mode": mode}

def _next_txn_id(df, prefix="TXN"):
    """Transaction ledger -> next sequential id string."""
    if df.empty:
        return f"{prefix}001"
    nums = [int(re.findall(r"(\d+)$", str(v))[0] or 0) if re.findall(r"(\d+)$", str(v)) else 0
            for v in df["transaction_id"].astype(str)]
    return f"{prefix}{max(nums) + 1:03d}"

def t_update_inventory(inventory_df, item_id, quantity_new=None, delta=None):
    """Set stock by item_id (delta or absolute) -> {inventory_df, updated} or {error}."""
    if item_id is None:
        return {"error": "item_id_missing"}
    inv = inventory_df.copy()
    inv["item_id"] = inv["item_id"].astype(str)
    mask = inv["item_id"] == str(item_id)
    if not mask.any():
        return {"error": "item_not_found"}
    current = int(inv.loc[mask, "quantity_in_stock"].iloc[0])
    if delta is None and quantity_new is None:
        return {"error": "need_delta_or_quantity_new"}
    new_q = int(quantity_new) if quantity_new is not None else current + int(delta)
    inv.loc[mask, "quantity_in_stock"] = new_q
    return {"inventory_df": inv,
            "updated": {"item_id": item_id, "quantity_in_stock": int(new_q)}}

def t_append_transaction(transaction_df, customer_name, summary, amount, txn_prefix="TXN"):
    """Append a ledger row, recomputing the running balance -> {transaction_df, transaction}."""
    out = transaction_df.copy()
    last_bal = float(out["balance_after_transaction"].iloc[-1]) if not out.empty else 0.0
    row = {
        "transaction_id": _next_txn_id(out, txn_prefix),
        "customer_name": customer_name,
        "transaction_summary": summary,
        "transaction_amount": float(amount),
        "balance_after_transaction": last_bal + float(amount),
    }
    out = pd.concat([out, pd.DataFrame([row])], ignore_index=True)
    return {"transaction_df": out, "transaction": row}

def t_compute_total(qty, price):
    """(qty, unit price) -> {amount} for a purchase."""
    return {"amount": float(qty) * float(price)}

def t_compute_refund(qty, price):
    """(qty, unit price) -> {amount}, negative by design for a return."""
    return {"amount": -float(qty) * float(price)}

def t_assert_true(value):
    """Truthy check -> {ok}."""
    return {"ok": bool(value)}

def t_assert_nonnegative_stock(inventory_df, item_id):
    """Stock level check after an update -> {ok, qty}."""
    mask = inventory_df["item_id"].astype(str) == str(item_id)
    if not mask.any():
        return {"ok": False, "reason": "item_not_found"}
    q = int(inventory_df.loc[mask, "quantity_in_stock"].iloc[0])
    return {"ok": q >= 0, "qty": q}

TOOL_REGISTRY = {
    # lookup_product / assert: aliases absorbing near-miss tool names from model plans.
    "get_inventory_data":       lambda **kw: t_get_inventory_data(kw["con"], kw.get("product_name"), kw.get("item_id")),
    "get_transaction_data":     lambda **kw: t_get_transaction_data(kw["con"], kw.get("mode", "last_balance")),
    "lookup_product":           lambda **kw: t_get_inventory_data(kw["con"], kw.get("product_name"), kw.get("item_id")),
    "update_inventory":         lambda **kw: t_update_inventory(kw["inventory_df"], kw["item_id"], kw.get("quantity_new"), kw.get("delta")),
    "append_transaction":       lambda **kw: t_append_transaction(kw["transaction_df"], kw["customer_name"], kw["summary"], kw["amount"], kw.get("txn_prefix", "TXN")),
    "compute_total":            lambda **kw: t_compute_total(kw["qty"], kw["price"]),
    "compute_refund":           lambda **kw: t_compute_refund(kw["qty"], kw["price"]),
    "assert_true":              lambda **kw: t_assert_true(kw["value"]),
    "assert":                   lambda **kw: t_assert_true(kw["value"]),
    "assert_nonnegative_stock": lambda **kw: t_assert_nonnegative_stock(kw["inventory_df"], kw["item_id"]),
}

Model-written plans arrive with small inconsistencies — alias argument names
(`quantity` for `qty`), values that refer to earlier results, missing arguments. The
source lab hardens the registry with three layers: argument canonicalization,
context-path resolution (a plan value like `"context.prod.item.price"` reads from the
results of earlier steps), and a required-argument check that fails loudly.

In [ ]:
TOOL_SIGNATURES = {
    "get_inventory_data": [], "get_transaction_data": [], "lookup_product": [],
    "update_inventory": ["item_id"],
    "append_transaction": ["customer_name", "summary", "amount"],
    "compute_total": ["qty", "price"], "compute_refund": ["qty", "price"],
    "assert_true": ["value"], "assert": ["value"],
    "assert_nonnegative_stock": ["inventory_df", "item_id"],
}

def canonicalize_args(tool_name, args):
    """Alias argument names -> the canonical names the tools expect."""
    a = dict(args or {})
    if tool_name in ("lookup_product", "get_inventory_data") and "product_name" not in a:
        for alt in ("name", "product", "query"):
            if a.get(alt) is not None:
                a["product_name"] = a.pop(alt)
                break
    if tool_name in ("compute_total", "compute_refund"):
        if "qty" not in a and "quantity" in a:
            a["qty"] = a.pop("quantity")
        if "price" not in a and "unit_price" in a:
            a["price"] = a.pop("unit_price")
    if tool_name == "update_inventory":
        if "delta" not in a and "change" in a:
            a["delta"] = a.pop("change")
        if "quantity_new" not in a:
            for alt in ("new_quantity", "quantity", "qty_new"):
                if a.get(alt) is not None:
                    a["quantity_new"] = a.pop(alt)
                    break
    if tool_name == "append_transaction" and "summary" not in a and "transaction_summary" in a:
        a["summary"] = a.pop("transaction_summary")
    return a

def missing_required(tool_name, args):
    """(tool, args) -> list of required argument names that are absent."""
    missing = [k for k in TOOL_SIGNATURES.get(tool_name, [])
               if k not in args or args[k] is None]
    if tool_name == "update_inventory" and "delta" not in args and "quantity_new" not in args:
        missing.append("delta|quantity_new")
    return missing

def get_from_context(ctx, path, default=None):
    """Dotted 'context.' path -> value from the execution context."""
    if not isinstance(path, str) or not path.startswith("context."):
        return path
    cur = ctx
    for part in path.split(".")[1:]:
        if isinstance(cur, dict) and part in cur:
            cur = cur[part]
        else:
            return default
    return cur

def resolve_args(args, ctx):
    """Plan args -> concrete args, with 'context.' references and *_from keys resolved."""
    out = {}
    for k, v in (args or {}).items():
        if isinstance(v, str) and v.startswith("context."):
            out[k.replace("_from", "")] = get_from_context(ctx, v)
        else:
            out[k] = v
    return out

## 4. The Executor

The executor runs a plan step by step: for each step it runs the listed tools (write
tools return updated DataFrames, which the executor applies to the working copies and
re-registers in DuckDB), then runs the step's validations, and records everything in a
report. A failed validation aborts execution — a purchase that would drive stock
negative stops before the ledger is touched further.

*Do:* run the cell and read the executor's contract — it runs plans, it never invents them.


In [ ]:
def run_tools_for_step(step, ctx):
    """One plan step -> dict of tool results keyed by result_key; applies DataFrame updates."""
    results = {}
    for spec in step.get("tools", []):
        name, rkey = spec.get("use"), spec.get("result_key")
        if not name or not rkey:
            raise ValueError("Each tool spec requires 'use' and 'result_key'")
        fn = TOOL_REGISTRY.get(name)
        if not fn:
            raise ValueError(f"Unknown tool: {name}")
        args = canonicalize_args(name, resolve_args(spec.get("args", {}), ctx))
        missing = missing_required(name, args)
        if missing:
            raise ValueError(f"Missing required args for tool '{name}': {missing}")
        args.setdefault("con", ctx["__con__"])
        args.setdefault("inventory_df", ctx["__frames__"]["inventory_df"])
        args.setdefault("transaction_df", ctx["__frames__"]["transaction_df"])
        res = fn(**args)
        if isinstance(res, dict):
            for frame in ("inventory_df", "transaction_df"):
                if isinstance(res.get(frame), pd.DataFrame):
                    ctx["__frames__"][frame] = res[frame]
                    ctx["__con__"].unregister(frame)
                    ctx["__con__"].register(frame, res[frame])
        ctx[rkey] = res
        results[rkey] = res
    return results

def run_tool_validation(v, ctx):
    """One validation spec -> {name, ok, ...}."""
    name, tname = v.get("name", "validation"), v.get("use_tool")
    fn = TOOL_REGISTRY.get(tname)
    if not fn:
        return {"name": name, "ok": False, "error": f"unknown_tool:{tname}"}
    args = canonicalize_args(tname, resolve_args(v.get("args", {}), ctx))
    missing = missing_required(tname, args)
    if missing:
        return {"name": name, "ok": False, "error": f"missing_required_args:{missing}"}
    args.setdefault("con", ctx["__con__"])
    args.setdefault("inventory_df", ctx["__frames__"]["inventory_df"])
    args.setdefault("transaction_df", ctx["__frames__"]["transaction_df"])
    res = fn(**args)
    return {"name": name, "ok": bool(res.get("ok", True)), "result": res}

def execute_plan_tools_only(plan, inventory_df, transaction_df,
                            return_updated_frames=True, stop_on_failed_validation=True):
    """Tools-only plan + current frames -> execution report (with updated frames)."""
    con = create_duckdb_with_views(inventory_df, transaction_df)
    ctx = {"__con__": con,
           "__frames__": {"inventory_df": inventory_df.copy(),
                          "transaction_df": transaction_df.copy()}}
    report = {"ok": True, "steps": []}
    try:
        for step in plan.get("steps", []):
            tool_error = None
            try:
                ran = run_tools_for_step(step, ctx)
            except Exception as e:
                ran, tool_error = {}, str(e)
                report["ok"] = False
            validations = [run_tool_validation(v, ctx) for v in step.get("validations", [])]
            if tool_error is not None or not all(v.get("ok", False) for v in validations):
                report["ok"] = False
            report["steps"].append({
                "step_number": step.get("step_number"),
                "description": step.get("description", ""),
                "tools_run": list(ran.keys()),
                "tool_error": tool_error,
                "validations": validations,
            })
            if stop_on_failed_validation and any(not v.get("ok", False) for v in validations):
                report.update(aborted=True, abort_step=step.get("step_number"),
                              abort_reason="validation_failed")
                break
    finally:
        con.close()
    if return_updated_frames:
        report["updated_frames"] = {
            "inventory_df": ctx["__frames__"]["inventory_df"],
            "transaction_df": ctx["__frames__"]["transaction_df"],
        }
    return report

A minimal hand-written plan exercises the executor before any model is involved: one
step, one lookup tool, one validation.

In [ ]:
simple_plan = {
    "reasoning": "User wants to check availability of Aviator sunglasses.",
    "steps": [
        {
            "step_number": 1,
            "description": "Lookup Aviator sunglasses in inventory",
            "tools": [
                {"use": "get_inventory_data", "args": {"product_name": "Aviator"},
                 "result_key": "prod"}
            ],
            "validations": [
                {"name": "product_found", "use_tool": "assert_true",
                 "args": {"value_from": "context.prod.item"}}
            ],
        }
    ],
}

report = execute_plan_tools_only(simple_plan, inventory_df, transaction_df)
print("ok:", report["ok"])
for step in report["steps"]:
    print(step["step_number"], step["description"], "| tools:", step["tools_run"],
          "| validations:", [(v["name"], v["ok"]) for v in step["validations"]])

The report shows the executed tools and the validation verdicts. Every model-generated
plan below runs through exactly this machinery; the model never touches the tables
directly.

## 5. The Pipeline's Agents

### 5.1 Planning step

The planning step turns a customer request into a full tools-only plan. Its
specification — the tool catalog with exact argument names, the strict JSON output
shape, and one worked example — is the source lab's, verbatim. The catalog text is
the planner's only knowledge of the tools, so argument names in it must match the
registry exactly (the W3 lab measured what happens when they do not).
*Do:* skim the spec's three parts — tool catalog, STRICT RULES, worked example — before running the cell.


In [ ]:
PLANNING_SPEC_TOOLS_ONLY = """
You are a planning system for a sunglasses store. Produce a FULL, AUTONOMOUS plan using TOOLS ONLY.
We will run this plan against two pandas DataFrames registered in DuckDB as views:
- inventory_df(name, item_id, description, quantity_in_stock, price)
- transaction_df(transaction_id, customer_name, transaction_summary, transaction_amount, balance_after_transaction)

Customer intents include:
- Purchase: "I want to buy 3 Aviators"
- Return: "I'd like to return two Sport sunglasses"
- Inquiry: "Do you have Mystique glasses?"
- Browse: "Show me what's available"

IMPORTANT: ALLOWED TOOLS ONLY (do NOT invent new tools)
Tool catalog (names, exact args, outputs):
1) get_inventory_data
   - args: { product_name?: string, item_id?: string }
   - returns: { rows: DataFrame, match_count: int, item: dict|null }
   - notes: Use this for product lookup (case-insensitive by name) or by item_id. No args = full catalog (browse).
2) get_transaction_data
   - args: { mode?: "last_balance" }
   - returns: { mode: string, last_txn_id: string|null, last_balance: number }
3) compute_total
   - args: { qty: number, price: number }
   - returns: { amount: number }
4) compute_refund
   - args: { qty: number, price: number }   # Refund is negative by design
   - returns: { amount: number }
5) update_inventory
   - args: { item_id: string, delta?: number, quantity_new?: number }
   - returns: { inventory_df: DataFrame, updated: { item_id: string, quantity_in_stock: number } }
   - notes: For purchase use delta = -qty. For return use delta = +qty.
6) append_transaction
   - args: { customer_name: string, summary: string, amount: number }
   - returns: { transaction_df: DataFrame, transaction: { ... } }
7) assert_true
   - args: { value: any }                    # passes if truthy (non-null/non-zero/non-empty)
   - returns: { ok: boolean }
8) assert_nonnegative_stock
   - args: { inventory_df: DataFrame, item_id: string }
   - returns: { ok: boolean, qty: number }

STRICT RULES:
1) Return VALID JSON ONLY with keys: reasoning, steps.
2) Each step MUST contain:
   - "step_number": integer
   - "description": short human text
   - "tools": an array of tool calls in order. Each tool call is:
       {"use": "<tool_name>", "args": {...}, "result_key": "<context_key>"}
     * You MAY reference previous results using dotted paths starting with "context.", e.g., "context.prod.item.price".
     * Use *_from to resolve from context, e.g., {"price_from": "context.prod.item.price"}.
     * Use ONLY the tools listed above. Do NOT use names like assert_one, assert_gt, assert_contains, format_return_summary, lookup_product, propose_transaction, etc.
     * Strings like the transaction summary MUST be composed inline by you (e.g., "Return 2 Sport sunglasses").
   - "validations": array of tool validations:
       {"name": "...", "use_tool": "<tool_name>", "args": {...}}
     * Allowed validation tools: assert_true, assert_nonnegative_stock ONLY.
     * Examples:
         - product_found: assert_true with {"value_from": "context.prod.item"} (non-null)
         - nonnegative_stock_after_update: assert_nonnegative_stock with {"inventory_df_from": "context.__frames__.inventory_df", "item_id_from": "context.prod.item.item_id"}
3) Do NOT include raw SQL in the plan. Tools run any needed SQL internally.
4) For purchases/returns, include tool calls to:
   - Lookup product via get_inventory_data (case-insensitive by name)
   - (Purchase only) compute_total; (Return only) compute_refund
   - Create a clear summary STRING inline (e.g., "Purchase 3 Aviator sunglasses" / "Return 2 Sport sunglasses")
   - Update inventory via update_inventory (delta = -qty for purchases, +qty for returns)
   - Append the transaction via append_transaction (amount from compute_total/compute_refund)
   - For purchases, validate stock with assert_nonnegative_stock AFTER the update.
5) For inquiry/browse requests, ONLY look up data; do NOT update inventory or append transactions.
6) Use canonical arg names exactly as in the Tool catalog:
   - quantity -> use qty
   - unit_price -> use price
   - Do NOT add extra args like sign; compute_refund already returns negative amounts.

OUTPUT JSON SHAPE:
{
  "reasoning": "...",
  "steps": [
    {
      "step_number": 1,
      "description": "...",
      "tools": [ {"use": "...", "args": {...}, "result_key": "..."} ],
      "validations": [ {"name": "...", "use_tool": "...", "args": {...}} ]
    }
  ]
}

EXAMPLE (Return 2 Sport sunglasses for a walk-in):
{
  "reasoning": "User requests a return of 2 Sport units. We'll lookup product, compute a negative refund, update stock (+2), and append a refund transaction.",
  "steps": [
    {
      "step_number": 1,
      "description": "Lookup product 'Sport' and capture item details",
      "tools": [
        {"use": "get_inventory_data", "args": {"product_name": "Sport"}, "result_key": "prod"}
      ],
      "validations": [
        {"name": "product_found", "use_tool": "assert_true", "args": {"value_from": "context.prod.item"}}
      ]
    },
    {
      "step_number": 2,
      "description": "Compute refund amount for qty=2",
      "tools": [
        {"use": "compute_refund", "args": {"qty": 2, "price_from": "context.prod.item.price"}, "result_key": "refund"}
      ],
      "validations": []
    },
    {
      "step_number": 3,
      "description": "Update inventory by adding returned quantity",
      "tools": [
        {"use": "update_inventory", "args": {"item_id_from": "context.prod.item.item_id", "delta": 2}, "result_key": "inv_after"}
      ],
      "validations": [
        {"name": "stock_nonnegative", "use_tool": "assert_nonnegative_stock",
         "args": {"inventory_df_from": "context.__frames__.inventory_df", "item_id_from": "context.prod.item.item_id"}}
      ]
    },
    {
      "step_number": 4,
      "description": "Append the refund transaction for a walk-in customer",
      "tools": [
        {"use": "append_transaction",
         "args": {
           "customer_name": "WALK_IN_CUSTOMER",
           "summary": "Return 2 Sport sunglasses",
           "amount_from": "context.refund.amount"
         },
         "result_key": "txn"}
      ],
      "validations": [
        {"name": "transaction_created", "use_tool": "assert_true",
         "args": {"value_from": "context.txn.transaction.transaction_id"}}
      ]
    }
  ]
}
"""

def generate_plan(user_query):
    """Customer request text -> tools-only plan dict."""
    context = f"{PLANNING_SPEC_TOOLS_ONLY}\n\nCustomer query: {user_query}\nProduce the plan now."
    reply = chat_text(context,
                      system="Return JSON ONLY following the TOOLS-ONLY planning spec.")
    return parse_json_output(reply)

In [ ]:
user_query = "I'd like to return two Aviator sunglasses"

draft_plan = generate_plan(user_query)
print(json.dumps(draft_plan, indent=2, default=str)[:2000])

The draft plan names tools from the catalog, threads values through `context.` paths,
and attaches validations. Drafts are not always spec-compliant — the next step exists
because of that.

### 5.2 Reflection step ✍️

A draft plan can carry alias argument names, missing validations, or malformed JSON.
The **reflection step** = a second model call that critiques the draft against the
specification and returns a corrected plan (the pattern built in the W5 lab, here
applied to a plan instead of code).

Write `REVIEWER_SYSTEM`, the reflection agent's system prompt. Requirements: it must
assign the reviewer role, demand STRICT JSON with exactly the keys `critique` (string)
and `revised_plan` (object), and require the revised plan to follow the tools-only
spec.

In [ ]:
### FILL IN (START) ###
REVIEWER_SYSTEM = (
    ""
)
### FILL IN (END) ###

def reflect_on_plan(user_query, draft_plan):
    """(request, draft plan) -> {critique, revised_plan}; falls back to the draft."""
    user = (
        "TOOLS-ONLY PLANNING SPEC (enforce exactly):\n"
        f"{PLANNING_SPEC_TOOLS_ONLY}\n\n"
        "Customer query:\n"
        f"{user_query}\n\n"
        "Draft plan (JSON):\n"
        f"{json.dumps(draft_plan, ensure_ascii=False)}\n\n"
        "Task: Critique the draft against the spec and return a corrected "
        "'revised_plan' if needed. Ensure valid JSON and that no raw SQL appears "
        "in the plan (only tool calls)."
    )
    try:
        data = parse_json_output(chat_text(user, system=REVIEWER_SYSTEM))
    except (ValueError, json.JSONDecodeError):
        return {"critique": "Unparseable reflection output; falling back to draft.",
                "revised_plan": draft_plan}
    if "revised_plan" not in data or not isinstance(data["revised_plan"], dict):
        if "steps" in data:
            return {"critique": "No explicit critique provided.", "revised_plan": data}
        return {"critique": "Malformed reflection output; falling back to draft.",
                "revised_plan": draft_plan}
    return data

A deliberately imperfect draft tests the step: a wrong argument name (`quantity`
instead of `qty`) and a missing `product_found` validation, as in the source lab.

In [ ]:
flawed_draft = {
    "reasoning": "User wants to buy 2 Aviators.",
    "steps": [
        {
            "step_number": 1,
            "description": "Lookup product 'Aviator'",
            "tools": [
                {"use": "get_inventory_data", "args": {"product_name": "Aviator"},
                 "result_key": "prod"}
            ],
            "validations": [],   # missing product_found validation
        },
        {
            "step_number": 2,
            "description": "Compute total for purchase",
            "tools": [
                {"use": "compute_total",   # wrong arg name: should be qty
                 "args": {"quantity": 2, "price_from": "context.prod.item.price"},
                 "result_key": "total"}
            ],
            "validations": [],
        },
    ],
}

reflection = reflect_on_plan("I'd like to buy 2 Aviator sunglasses for a walk-in customer",
                             flawed_draft)
print("CRITIQUE:\n", reflection["critique"])
print("\nREVISED PLAN:\n", json.dumps(reflection["revised_plan"], indent=2, default=str)[:1500])

The critique names the spec violations and the revised plan repairs them. When the
reflection output itself is malformed, `reflect_on_plan` falls back to the draft — a
pipeline stage must degrade, not crash.

### 5.3 Error-explanation step ✍️

Execution can still fail — insufficient stock, an unknown product. The report that
records the failure is machine-shaped JSON; the **error-explanation step** = a model
call that turns the failed report into guidance a store clerk could act on.

Write `EXPLAINER_SYSTEM`, this agent's system prompt. Requirements: assign an error-explainer role (a support
assistant addressing a store clerk), demand a plain-language explanation of what went wrong and how to fix it, and
forbid technical jargon (no JSON keys, no tool names).

In [ ]:
### FILL IN (START) ###
EXPLAINER_SYSTEM = (
    ""
)
### FILL IN (END) ###

def explain_execution_error(user_query, execution_report):
    """(request, failed report) -> plain-language explanation text."""
    user = (
        "Customer query:\n"
        f"{user_query}\n\n"
        "Execution report (JSON):\n"
        f"{json.dumps(execution_report, default=str)}\n\n"
        "Task: Explain in simple terms what went wrong and how to fix it."
    )
    return chat_text(user, system=EXPLAINER_SYSTEM)

In [ ]:
failed_report = {
    "ok": False,
    "steps": [
        {"step_number": 1, "description": "Lookup product 'Aviator'",
         "tools_run": ["get_inventory_data"], "tool_error": None,
         "validations": [{"name": "product_found", "ok": True}]},
        {"step_number": 2, "description": "Update inventory by subtracting 999 units",
         "tools_run": ["update_inventory"], "tool_error": None,
         "validations": [{"name": "stock_nonnegative", "ok": False, "qty": -995}]},
    ],
    "aborted": True, "abort_step": 2, "abort_reason": "validation_failed",
}

print(explain_execution_error("Buy 999 Aviator sunglasses", failed_report))

The explanation restates the failure in stock-and-purchase terms. This step runs only
when `report["ok"]` is false; successful requests need no explanation call.

## 6. Pipeline Assembly

`handle_request` chains the stages: plan → reflect → execute → (explain on failure).
Updated tables are applied only when execution succeeded; a failed request leaves the
store's data untouched.

*Do:* run the cell on the sample request; the verdict, the full per-stage report,
and the updated tables print below.


In [ ]:
def handle_request(user_query, inventory_df, transaction_df, use_reflection=True):
    """Request + current frames -> (report, new inventory, new transactions, message|None)."""
    draft = generate_plan(user_query)
    plan = reflect_on_plan(user_query, draft)["revised_plan"] if use_reflection else draft
    report = execute_plan_tools_only(plan, inventory_df, transaction_df)
    if report["ok"]:
        frames = report["updated_frames"]
        return report, frames["inventory_df"], frames["transaction_df"], None
    message = explain_execution_error(user_query, report)
    return report, inventory_df, transaction_df, message

In [ ]:
request = "I'd like to return two Aviator sunglasses for a walk-in customer"

report, inv_after, txn_after, message = handle_request(request, inventory_df, transaction_df)
print("ok:", report["ok"])
print(json.dumps(report, indent=2, default=str)[:1200])   # the full handoff record
display(inv_after[["name", "quantity_in_stock"]])
display(txn_after.tail(2))
if message:
    print(message)

The end-to-end run shows the division of labor: the planner wrote the plan, the
reviewer repaired it, the executor changed the tables under validation, and no single
model call ever held more than its own role's context.

## 7. Measured Task — A Day of Customer Requests ✍️ (core)

Six requests cover the store's intents: purchases, returns, inquiries, browsing, and
one impossible order that must fail safely. For each request the scorer runs the full
pipeline on fresh tables and checks the outcome programmatically: the report's
success flag, the stock change for the affected product, and whether a ledger row was
appended.

**Target: ≥ 5/6.** Most of the remaining time belongs here: read the failing
request's report and plan, adjust the fill-in prompts (or, if the plan itself is at
fault, the planning spec's rules), and re-run — one change at a time.

In [ ]:
#@title Taskset and scorer — run as-is (six requests + programmatic outcome checks) { display-mode: "form" }
TASKSET = [
    {"query": "I'd like to buy 2 Aviator sunglasses",
     "item_id": "SG001", "delta": -2, "ok": True,  "txn": True},
    {"query": "I'd like to return two Sport sunglasses for a walk-in customer",
     "item_id": "SG004", "delta": +2, "ok": True,  "txn": True},
    {"query": "Do you have Mystique glasses in stock?",
     "item_id": "SG003", "delta": 0,  "ok": True,  "txn": False},
    {"query": "Show me everything you have available",
     "item_id": None,    "delta": 0,  "ok": True,  "txn": False},
    {"query": "I want to buy 999 Wayfarer sunglasses",
     "item_id": "SG002", "delta": 0,  "ok": False, "txn": False},
    {"query": "I'd like to return one Round sunglasses for a walk-in customer",
     "item_id": "SG005", "delta": +1, "ok": True,  "txn": True},
]

def stock_of(inv, item_id):
    """(inventory, item_id) -> current stock quantity."""
    return int(inv.loc[inv["item_id"] == item_id, "quantity_in_stock"].iloc[0])

def score_pipeline(use_reflection=True):
    """Run TASKSET on fresh tables -> number of requests handled correctly."""
    correct = 0
    for task in TASKSET:
        inv, txn = create_inventory_dataframe(), create_transaction_dataframe()
        before = stock_of(inv, task["item_id"]) if task["item_id"] else None
        try:
            report, inv2, txn2, _ = handle_request(task["query"], inv, txn,
                                                   use_reflection=use_reflection)
        except Exception as exc:
            print(f"FAIL  {task['query'][:48]:50} pipeline error: {exc}")
            continue
        checks = [report["ok"] == task["ok"],
                  len(txn2) - len(txn) == (1 if task["txn"] else 0)]
        if task["item_id"]:
            checks.append(stock_of(inv2, task["item_id"]) - before == task["delta"])
        ok = all(checks)
        correct += ok
        print(f"{'PASS' if ok else 'FAIL'}  {task['query'][:48]:50} "
              f"report_ok={report['ok']} txn_rows=+{len(txn2) - len(txn)}")
    print(f"SCORE: {correct}/{len(TASKSET)}")
    return correct

TARGET_SCORE = 5
pipeline_score = score_pipeline()
print("TARGET REACHED" if pipeline_score >= TARGET_SCORE else "KEEP ITERATING")

A failing purchase or return usually traces to the plan (wrong delta sign, missing
`append_transaction`) or to a reflection output that replaced a valid draft with a
broken one — the per-step report says which. The impossible order must fail at the
`assert_nonnegative_stock` validation, not succeed and not crash.

To read one request's handoffs in full, re-run it alone on fresh copies of the
tables:

In [ ]:
# Debug one request in isolation (edit the index into TASKSET).
dbg_report, dbg_inv, dbg_txn, dbg_msg = handle_request(
    TASKSET[4]["query"], inventory_df, transaction_df)
print(json.dumps(dbg_report, indent=2, default=str)[:1500])
if dbg_msg:
    print("\nexplainer:", dbg_msg)

## 8. Exercises ✍️

Protocol: write the prediction down first, run the cell, compare.

### 8.1 Reflection ablation

The variant below skips the reflection step and executes draft plans directly.
Prediction: the score with reflection was measured above — does removing the step
change the score, and how many model calls does it save per request?

In [ ]:
n_calls = 0
score_no_reflection = score_pipeline(use_reflection=False)
calls_without = n_calls

n_calls = 0
score_with_reflection = score_pipeline(use_reflection=True)
calls_with = n_calls

print(f"\nwith reflection: {score_with_reflection}/6 in {calls_with} calls; "
      f"without: {score_no_reflection}/6 in {calls_without} calls")

When the planner already produces spec-compliant plans, the reviewer adds calls
without adding correctness; when drafts are flawed (as in Section 5.2), it is the
difference between a repaired plan and a crashed step. The reflection stage is a
cost–reliability trade, priced in calls.

### 8.2 A new team member — customer-reply agent

The source course's second lab (M5_UGL_2, market-research team) chains specialized
roles — researcher, designer, copywriter, packager — each consuming the previous
role's output. The same extension applies here: a reply agent that turns the
execution outcome into a short customer-facing message. Prediction: what information
must this agent receive, and what must it never expose (internal ids, tool names)?

In [ ]:
REPLY_SYSTEM = (
    "You are a store assistant writing to a customer. In at most 3 sentences, "
    "confirm what happened with their request in plain, friendly language. "
    "Never mention internal identifiers, tool names, JSON, or system details."
)

def customer_reply_agent(user_query, report):
    """(request, execution report) -> short customer-facing reply."""
    user = (f"Customer request:\n{user_query}\n\n"
            f"Execution report (JSON):\n{json.dumps(report, default=str)}\n\n"
            "Write the reply now.")
    return chat_text(user, system=REPLY_SYSTEM)

ok_report, *_ = handle_request("I'd like to buy 2 Aviator sunglasses",
                               create_inventory_dataframe(), create_transaction_dataframe())
print("SUCCESS CASE:\n", customer_reply_agent("I'd like to buy 2 Aviator sunglasses", ok_report))

print("\nFAILURE CASE:\n", customer_reply_agent("Buy 999 Aviator sunglasses", failed_report))

Each role sees only what its task requires — the reply agent reads the report, not
the planning spec. Context separation is one of the two standard reasons for
splitting an agent into several (notes Ch. 6).

## 9. Completion Check

All rows must read `PASS` before submission; grading checks these same structural
facts, never prose quality.

In [ ]:
completion = {
    "REVIEWER_SYSTEM written (>= 40 chars)":  len(REVIEWER_SYSTEM.strip()) >= 40,
    "REVIEWER_SYSTEM names both keys":        "critique" in REVIEWER_SYSTEM
                                              and "revised_plan" in REVIEWER_SYSTEM,
    "EXPLAINER_SYSTEM written (>= 40 chars)": len(EXPLAINER_SYSTEM.strip()) >= 40,
    "pipeline ran end to end":                "pipeline_score" in dir(),
    "target score reached (>= 5/6)":          pipeline_score >= TARGET_SCORE,
}
for item, ok in completion.items():
    print(f"{'PASS' if ok else 'FAIL':4}  {item}")
print("\nLAB COMPLETE" if all(completion.values()) else "\nNOT COMPLETE YET")

---

W7 assembles the semester's largest pipeline: the research-agent workflow
(planner → research → writer → editor) from the *Agentic AI* final project.
Reference answers for this lab and the homework:
`labs/checkpoints/week06/solution.py`, published after the homework deadline.